In [2]:
import os
import pandas as pd

data_folder = "data"  # your folder path

# map table_name -> dataframe
tables = {}

for file in os.listdir(data_folder):
    if file.endswith(".csv"):
        table_name = file.replace(".csv", "")
        df = pd.read_csv(os.path.join(data_folder, file))
        tables[table_name] = df

In [3]:
import os
import pandas as pd

data_folder = "data"  # change this

table_column_dict = {}

for file in os.listdir(data_folder):
    if file.endswith(".csv"):
        table_name = file.replace(".csv", "")

        df = pd.read_csv(os.path.join(data_folder, file), nrows=5)
        # only reading few rows for speed

        table_column_dict[table_name] = df.columns.tolist()

# print nicely
import pprint
pprint.pprint(table_column_dict)

{'Album': ['AlbumId', 'Title', 'ArtistId'],
 'Artist': ['ArtistId', 'Name'],
 'Customer': ['CustomerId',
              'FirstName',
              'LastName',
              'Company',
              'Address',
              'City',
              'State',
              'Country',
              'PostalCode',
              'Phone',
              'Fax',
              'Email',
              'SupportRepId'],
 'Employee': ['EmployeeId',
              'LastName',
              'FirstName',
              'Title',
              'ReportsTo',
              'BirthDate',
              'HireDate',
              'Address',
              'City',
              'State',
              'Country',
              'PostalCode',
              'Phone',
              'Fax',
              'Email'],
 'Genre': ['GenreId', 'Name'],
 'Invoice': ['InvoiceId',
             'CustomerId',
             'InvoiceDate',
             'BillingAddress',
             'BillingCity',
             'BillingState',
             'Billing

In [4]:
embed_config = {
    "Album": ["Title"],

    "Artist": ["Name"],

    "Customer": [
        "FirstName",
        "LastName",
        "Company",
        "City",
        "State",
        "Country",
        "Email"
    ],

    "Employee": [
        "FirstName",
        "LastName",
        "Title",
        "City",
        "State",
        "Country",
        "Email"
    ],

    "Genre": ["Name"],

    "Invoice": [
        "BillingCity",
        "BillingState",
        "BillingCountry"
    ],

    "MediaType": ["Name"],

    "Playlist": ["Name"],

    "Track": [
        "Name",
        "Composer"
    ]
}

In [6]:
import os
os.environ["GOOGLE_API_KEY"] = os.getenv("GOOGLE_API_KEY")  # set this in your environment variables

In [7]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)


Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [8]:
from collections import defaultdict

documents = []
metadata = []
embed_texts = []

value_map = defaultdict(set)
column_map = defaultdict(set)   # 🔥 NEW (for grouping columns)

# 🔥 Step 1: Collect everything
for table, cols in embed_config.items():
    df = tables[table]

    # ✅ table-level
    text = f"{table} is a table in the database"
    documents.append(text)
    metadata.append({"type": "table", "table": table})
    embed_texts.append(table)

    for col in cols:
        # 🔥 collect column locations
        column_map[col].add(table)

        unique_vals = df[col].dropna().astype(str).str.lower().unique()[:100]

        for val in unique_vals:
            value_map[val].add((table, col))


# 🔥 Step 2: Create grouped column docs
for col, tables_list in column_map.items():

    tables_list = sorted(list(tables_list))

    # 🔥 format: Artist table, Genre table
    joined_tables = ", ".join([f"{t} table" for t in tables_list])

    text = f"{col} is a column in {joined_tables}"

    documents.append(text)
    metadata.append({
        "type": "column",
        "column": col,
        "tables": tables_list
    })

    # 🔥 better embedding (context-aware)
    embed_texts.append(f"{col} column")


# 🔥 Step 3: Create grouped value docs
for val, locations in value_map.items():

    locations = sorted(list(locations))

    parts = [f"{col} column of {table} table" for table, col in locations]

    joined = ", ".join(parts)

    full_text = f"{val} is present in {joined}"

    documents.append(full_text)
    metadata.append({
        "type": "value",
        "value": val,
        "locations": locations
    })

    # 🔥 ONLY VALUE goes to embedding
    embed_texts.append(val)

In [9]:
embed_texts[:5]

['Album', 'Artist', 'Customer', 'Employee', 'Genre']

In [10]:
len(embed_texts)

761

In [11]:
documents[:5]

['Album is a table in the database',
 'Artist is a table in the database',
 'Customer is a table in the database',
 'Employee is a table in the database',
 'Genre is a table in the database']

In [12]:
import time

batch_size = 20
embeddings = []

for i in range(0, len(embed_texts), batch_size):
    batch = embed_texts[i:i+batch_size]

    while True:
        try:
            emb = embeddings_model.embed_documents(batch)
            embeddings.extend(emb)

            print(f"Done {i + len(batch)} / {len(embed_texts)}")

            time.sleep(15)  # 🔥 VERY IMPORTANT
            break

        except Exception as e:
            print("⚠️ Rate limit hit, retrying in 20 sec...")
            time.sleep(20)

Done 20 / 761
Done 40 / 761
Done 60 / 761
Done 80 / 761
Done 100 / 761
Done 120 / 761
Done 140 / 761
Done 160 / 761
Done 180 / 761
Done 200 / 761
Done 220 / 761
Done 240 / 761
Done 260 / 761
Done 280 / 761
Done 300 / 761
Done 320 / 761
Done 340 / 761
Done 360 / 761
Done 380 / 761
Done 400 / 761
Done 420 / 761
Done 440 / 761
Done 460 / 761
Done 480 / 761
Done 500 / 761
Done 520 / 761
Done 540 / 761
Done 560 / 761
Done 580 / 761
Done 600 / 761
Done 620 / 761
Done 640 / 761
Done 660 / 761
Done 680 / 761
Done 700 / 761
Done 720 / 761
Done 740 / 761
Done 760 / 761
Done 761 / 761


In [13]:
import numpy as np

embeddings_np = np.array(embeddings).astype("float32")

In [14]:
import faiss

dim = embeddings_np.shape[1]

# L2 distance index
index = faiss.IndexFlatL2(dim)

# add vectors
index.add(embeddings_np)

print(f"✅ Stored {index.ntotal} vectors")

✅ Stored 761 vectors


In [16]:
faiss.write_index(index, r"RAG/vector.index")

In [17]:
import pickle

# save documents
with open(r"RAG\documents.pkl", "wb") as f:
    pickle.dump(documents, f)

# save metadata
with open(r"RAG\metadata.pkl", "wb") as f:
    pickle.dump(metadata, f)

# optional (debug / reuse)
with open(r"RAG\embed_texts.pkl", "wb") as f:
    pickle.dump(embed_texts, f)

In [18]:
print(len(documents))
print(len(metadata))
print(len(embeddings_np))

761
761
761


In [19]:
import faiss
import pickle
import numpy as np

# load FAISS index
index = faiss.read_index(r"RAG\vector.index")

# load stored data
with open(r"RAG\documents.pkl", "rb") as f:
    documents = pickle.load(f)

with open(r"RAG\metadata.pkl", "rb") as f:
    metadata = pickle.load(f)

In [20]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings

embeddings_model = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001"
)

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


In [21]:
query = "rocha"

query_embedding = embeddings_model.embed_query(query)

query_embedding = np.array([query_embedding]).astype("float32")
k = 5  # top results

distances, indices = index.search(query_embedding, k)
print("\n🔍 Top Results:\n")

for i, idx in enumerate(indices[0]):
    print(f"Rank {i+1}")
    print("Document:", documents[idx])
    print("Metadata:", metadata[idx])
    print("Distance:", distances[0][i])
    print("-" * 50)


🔍 Top Results:

Rank 1
Document: rocha is present in LastName column of Customer table
Metadata: {'type': 'value', 'value': 'rocha', 'locations': [('Customer', 'LastName')]}
Distance: 0.22704309
--------------------------------------------------
Rank 2
Document: rojas is present in LastName column of Customer table
Metadata: {'type': 'value', 'value': 'rojas', 'locations': [('Customer', 'LastName')]}
Distance: 0.53211963
--------------------------------------------------
Rank 3
Document: luisrojas@yahoo.cl is present in Email column of Customer table
Metadata: {'type': 'value', 'value': 'luisrojas@yahoo.cl', 'locations': [('Customer', 'Email')]}
Distance: 0.62442803
--------------------------------------------------
Rank 4
Document: brazil is present in Country column of Customer table, BillingCountry column of Invoice table
Metadata: {'type': 'value', 'value': 'brazil', 'locations': [('Customer', 'Country'), ('Invoice', 'BillingCountry')]}
Distance: 0.63780797
-----------------------